In [1]:
# Load required libraries and the clinical/technical CSVs
import os
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

# Try expected locations first; otherwise search under the project for the CSVs
# Default expected locations
clinical_path = DATA_DIR / "hvsmr_clinical.csv"
technical_path = DATA_DIR / "hvsmr_technical.csv"

# Repository layout (as discovered): data/diseased_data/
alt_clinical_path = DATA_DIR / "diseased_data" / "hvsmr_clinical.csv"
alt_technical_path = DATA_DIR / "diseased_data" / "hvsmr_technical.csv"

print("Project root:", PROJECT_ROOT)
print("Expected clinical path:", clinical_path)
print("Expected technical path:", technical_path)

# Prefer the discovered alternate paths if the defaults aren't present
if (not clinical_path.exists()) and alt_clinical_path.exists():
    clinical_path = alt_clinical_path
if (not technical_path.exists()) and alt_technical_path.exists():
    technical_path = alt_technical_path

# Final fallback: search under project root if still missing
if not clinical_path.exists() or not technical_path.exists():
    candidates = list(PROJECT_ROOT.rglob("hvsmr_*clinical*.csv")) + list(PROJECT_ROOT.rglob("hvsmr_*technical*.csv"))
    print("\nDid not find CSVs at expected paths. Candidate CSV files found:")
    for p in sorted(set(candidates)):
        print(" -", p)
    raise FileNotFoundError(
        f"Could not locate hvsmr_clinical.csv and/or hvsmr_technical.csv.\n"
        f"Tried: {DATA_DIR/'hvsmr_clinical.csv'} and {DATA_DIR/'hvsmr_technical.csv'} and also {alt_clinical_path} / {alt_technical_path}."
    )

hvsmr_clinical = pd.read_csv(clinical_path)
hvsmr_technical = pd.read_csv(technical_path)

print("hvsmr_clinical shape:", hvsmr_clinical.shape)
print("hvsmr_technical shape:", hvsmr_technical.shape)

display(hvsmr_clinical.head())
display(hvsmr_technical.head())

print("Clinical columns:", list(hvsmr_clinical.columns))
print("Technical columns:", list(hvsmr_technical.columns))

Project root: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026
Expected clinical path: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/data/hvsmr_clinical.csv
Expected technical path: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/data/hvsmr_technical.csv
hvsmr_clinical shape: (59, 36)
hvsmr_technical shape: (137, 7)


,Pat,Age,Category,Normal,MildModerateDilation,VSD,ASD,DORV,DLoopTGA,ArterialSwitch,...,Glenn,Fontan,Heterotaxy,SuperoinferiorVentricles,PAAtresiaOrMPAStump,PABanding,AOPAAnastamosis,Marfan,CMRArtifactAO,CMRArtifactPA
0,0,10,moderate,NaN,NaN,X,NaN,X,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,X,NaN,NaN,NaN,NaN
1,1,11,mild,X,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,16,mild,X,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,X,NaN,NaN
3,3,52,severe,NaN,NaN,NaN,X,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,X
4,4,2,severe,NaN,NaN,X,X,X,NaN,NaN,...,NaN,NaN,NaN,NaN,X,NaN,NaN,NaN,NaN,NaN


,Pat,TE,TR,FA,BW,HVSMR2016,PaceMEDIA2022
0,0.0,2.1,4.1,75.0,755.0,train,1
1,1.0,1.7,3.4,55.0,1040.0,train,4
2,2.0,1.6,3.3,55.0,1040.0,train,2
3,3.0,1.5,3.1,55.0,1040.0,train,2
4,4.0,2.4,4.9,90.0,540.0,train,4


Clinical columns: ['Pat', 'Age', 'Category', 'Normal', 'MildModerateDilation', 'VSD', 'ASD', 'DORV', 'DLoopTGA', 'ArterialSwitch', 'BilateralSVC', 'SevereDilation', 'TortuousVessels', 'Dextrocardia', 'Mesocardia', 'InvertedVentricles', 'InvertedAtria', 'LeftCentralIVC', 'LeftCentralSVC', 'LLoopTGA', 'AtrialSwitch', 'Rastelli', 'SingleVentricle', 'DILV', 'DIDORV', 'CommonAtrium', 'Glenn', 'Fontan', 'Heterotaxy', 'SuperoinferiorVentricles', 'PAAtresiaOrMPAStump', 'PABanding', 'AOPAAnastamosis', 'Marfan', 'CMRArtifactAO', 'CMRArtifactPA']
Technical columns: ['Pat', 'TE', 'TR', 'FA', 'BW', 'HVSMR2016', 'PaceMEDIA2022']


In [2]:
# Build a mapping from Pat -> segmentation filepath for *_cropped_seg_endpoints.nii.gz
from pathlib import Path

cropped_norm_dir = DATA_DIR / "diseased_data" / "cropped_norm"
print("cropped_norm_dir:", cropped_norm_dir)
print("exists:", cropped_norm_dir.exists())

seg_paths = sorted(cropped_norm_dir.rglob("*_cropped_seg_endpoints.nii.gz")) if cropped_norm_dir.exists() else []
print("Found segmentation files:", len(seg_paths))

# Helper to robustly parse patient id from filename patterns like "<Pat>_cropped_seg_endpoints.nii.gz"
def _parse_pat_from_name(p: Path):
    stem = p.name
    # remove known suffix
    suffix = "_cropped_seg_endpoints.nii.gz"
    if stem.endswith(suffix):
        prefix = stem[: -len(suffix)]
    else:
        prefix = p.stem
    # if prefix contains extra tokens, keep first token that is numeric
    tokens = prefix.replace("-", "_").split("_")
    for t in tokens:
        # common pattern in this dataset: "pat0", "pat10", ...
        if t.lower().startswith("pat") and t[3:].isdigit():
            return int(t[3:])
        if t.isdigit():
            return int(t)
    # last resort: try int cast of whole prefix
    try:
        return int(prefix)
    except Exception:
        return None

pat_to_seg = {}
ambiguous = []
for sp in seg_paths:
    pat = _parse_pat_from_name(sp)
    if pat is None:
        ambiguous.append(sp)
        continue
    if pat in pat_to_seg and pat_to_seg[pat] != sp:
        ambiguous.append(sp)
    else:
        pat_to_seg[pat] = sp

print("Mapped Pats:", len(pat_to_seg))
if ambiguous:
    print("Ambiguous/unparsed files (showing up to 10):")
    for p in ambiguous[:10]:
        print(" -", p)

# Compare against Pats in clinical + technical tables
clinical_pats = set(pd.to_numeric(hvsmr_clinical["Pat"], errors="coerce").dropna().astype(int).tolist())
technical_pats = set(pd.to_numeric(hvsmr_technical["Pat"], errors="coerce").dropna().astype(int).tolist())
mapped_pats = set(pat_to_seg.keys())

print("Clinical Pats:", len(clinical_pats), "Technical Pats:", len(technical_pats), "Mapped Pats:", len(mapped_pats))
print("Clinical missing seg:", len(clinical_pats - mapped_pats), "(show up to 15)", sorted(list(clinical_pats - mapped_pats))[:15])
print("Seg not in clinical:", len(mapped_pats - clinical_pats), "(show up to 15)", sorted(list(mapped_pats - clinical_pats))[:15])

# Show a few mappings
sample_items = list(pat_to_seg.items())[:10]
for pat, path in sample_items:
    print(f"Pat {pat}: {path}")

cropped_norm_dir: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/data/diseased_data/cropped_norm
exists: True
Found segmentation files: 60
Mapped Pats: 60
Clinical Pats: 59 Technical Pats: 60 Mapped Pats: 60
Clinical missing seg: 0 (show up to 15) []
Seg not in clinical: 1 (show up to 15) [59]
Pat 0: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/data/diseased_data/cropped_norm/pat0_cropped_seg_endpoints.nii.gz
Pat 10: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/data/diseased_data/cropped_norm/pat10_cropped_seg_endpoints.nii.gz
Pat 11: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/data/diseased_data/cropped_norm/pat11_cropped_seg_endpoints.nii.gz
Pat 12: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/data/diseased_data/cropped_norm/pat12_cropped_seg_endpoints.nii.gz
Pat 13: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/data/diseased_data/c

In [3]:
# Extract a surface mesh from one segmentation volume using marching cubes (sanity check)
# - Loads NIfTI (.nii.gz)
# - Inspects labels
# - Runs marching cubes on a chosen label

# Ensure NIfTI I/O dependency is available
try:
    import nibabel as nib
except ModuleNotFoundError:
    !pip -q install nibabel
    import nibabel as nib

# Ensure marching cubes dependency is available
try:
    from skimage.measure import marching_cubes
except ModuleNotFoundError:
    !pip -q install scikit-image
    from skimage.measure import marching_cubes

# pick a patient that exists in the mapping
sample_pat = sorted(pat_to_seg.keys())[0]
seg_path = pat_to_seg[sample_pat]
print("Sample Pat:", sample_pat)
print("Seg path:", seg_path)

img = nib.load(str(seg_path))
seg = img.get_fdata()
affine = img.affine
zooms = img.header.get_zooms()[:3]

print("Seg dtype:", seg.dtype)
print("Seg shape:", seg.shape)
print("Voxel spacing (zooms):", zooms)

# Basic label inspection
unique_vals = np.unique(seg)
print("Unique values (min/max/count):", unique_vals.min(), unique_vals.max(), unique_vals.size)
print("First up-to-20 unique values:", unique_vals[:20])

# Choose a non-zero label if present (common for segmentations)
nonzero = unique_vals[unique_vals != 0]
if nonzero.size == 0:
    raise ValueError("Segmentation appears to be all zeros; cannot extract a surface.")

label = float(nonzero[0])
print("Using label for surface extraction:", label)

mask = (seg == label).astype(np.uint8)
print("Mask voxels:", int(mask.sum()))

# Run marching cubes in voxel space; provide spacing to get vertices in mm
verts, faces, normals, values = marching_cubes(mask, level=0.5, spacing=zooms)
print("Extracted mesh: verts", verts.shape, "faces", faces.shape)

# Show a few vertices/faces
print("Verts sample:\n", verts[:5])
print("Faces sample:\n", faces[:5])


Sample Pat: 0
Seg path: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/data/diseased_data/cropped_norm/pat0_cropped_seg_endpoints.nii.gz
Seg dtype: float64
Seg shape: (127, 207, 141)
Voxel spacing (zooms): (0.885417, 0.885417, 0.889999)
Unique values (min/max/count): 0.0 8.0 6
First up-to-20 unique values: [0. 3. 5. 6. 7. 8.]
Using label for surface extraction: 3.0
Mask voxels: 5953
Extracted mesh: verts (4626, 3) faces (9248, 3)
Verts sample:
 [[72.16148  60.208355 43.60995 ]
 [72.604195 60.208355 43.16495 ]
 [72.604195 59.765648 43.60995 ]
 [72.604195 59.765648 44.49995 ]
 [72.16148  60.208355 44.49995 ]]
Faces sample:
 [[2 1 0]
 [4 3 0]
 [0 3 2]
 [4 5 3]
 [0 1 6]]


In [4]:
# Extract surface meshes for all patients using marching cubes (per label)
import numpy as np
from skimage.measure import marching_cubes

# Choose which segmentation label(s) to extract as surfaces.
# We'll extract a mesh for each non-zero label present in each patient.

def extract_meshes_from_seg(seg_nii_path, labels="all_nonzero"):
    """Return dict: label -> (verts_mm, faces) using marching cubes on binary mask."""
    img = nib.load(str(seg_nii_path))
    seg = img.get_fdata()
    zooms = img.header.get_zooms()[:3]

    u = np.unique(seg)
    nonzero = u[u != 0]
    if labels == "all_nonzero":
        use_labels = nonzero
    else:
        use_labels = np.array(labels, dtype=float)

    out = {}
    for lab in use_labels:
        mask = (seg == float(lab)).astype(np.uint8)
        if mask.sum() < 10:
            continue
        verts, faces, normals, values = marching_cubes(mask, level=0.5, spacing=zooms)
        out[float(lab)] = (verts.astype(np.float32), faces.astype(np.int32))
    return out

# Batch extract
pat_meshes = {}  # pat -> dict(label -> (verts, faces))
mesh_qc = []

for pat, sp in sorted(pat_to_seg.items()):
    meshes = extract_meshes_from_seg(sp, labels="all_nonzero")
    pat_meshes[pat] = meshes
    # QC summary
    labs = sorted(meshes.keys())
    n_total_verts = int(sum(meshes[l][0].shape[0] for l in labs))
    n_total_faces = int(sum(meshes[l][1].shape[0] for l in labs))
    mesh_qc.append((pat, len(labs), labs[:10], n_total_verts, n_total_faces))

mesh_qc_df = pd.DataFrame(mesh_qc, columns=["Pat", "n_labels", "labels_sample", "total_verts", "total_faces"])
display(mesh_qc_df.head(10))
print("Patients with >=1 extracted mesh:", int((mesh_qc_df['n_labels'] > 0).sum()), "/", mesh_qc_df.shape[0])
print("Label count distribution:\n", mesh_qc_df['n_labels'].value_counts().sort_index())


,Pat,n_labels,labels_sample,total_verts,total_faces
0,0,5,"[3.0, 5.0, 6.0, 7.0, 8.0]",31665,62914
1,1,5,"[3.0, 5.0, 6.0, 7.0, 8.0]",27594,54812
2,2,5,"[3.0, 5.0, 6.0, 7.0, 8.0]",26532,52634
3,3,5,"[3.0, 5.0, 6.0, 7.0, 8.0]",30517,60550
4,4,5,"[3.0, 5.0, 6.0, 7.0, 8.0]",4783,9240
5,5,5,"[3.0, 5.0, 6.0, 7.0, 8.0]",11461,22558
6,6,5,"[3.0, 5.0, 6.0, 7.0, 8.0]",23030,45598
7,7,5,"[3.0, 5.0, 6.0, 7.0, 8.0]",34145,67854
8,8,5,"[3.0, 5.0, 6.0, 7.0, 8.0]",24422,48604
9,9,5,"[3.0, 5.0, 6.0, 7.0, 8.0]",39336,78192


Patients with >=1 extracted mesh: 60 / 60
Label count distribution:
 n_labels
4     3
5    57
Name: count, dtype: int64


In [5]:
# Load the normal reference mesh from OBJ and inspect basic geometry
from pathlib import Path

obj_path = PROJECT_ROOT / "SubTool-0-7412864.OBJ"
print("OBJ path:", obj_path)
print("Exists:", obj_path.exists())

if not obj_path.exists():
    # Fallback: search under project root
    candidates = list(PROJECT_ROOT.rglob("SubTool-0-7412864.OBJ")) + list(PROJECT_ROOT.rglob("*.obj")) + list(PROJECT_ROOT.rglob("*.OBJ"))
    print("\nCould not find OBJ at expected path. Candidate OBJ files (showing up to 50):")
    for p in sorted(set(candidates))[:50]:
        print(" -", p)
    raise FileNotFoundError("Could not locate SubTool-0-7412864.OBJ")

# Simple OBJ parser (supports 'v' and 'f' lines; triangulates polygons via fan)
# Note: faces in OBJ are 1-indexed; can be 'v', 'v/vt', or 'v/vt/vn'

def load_obj_vertices_faces(path: Path):
    verts = []
    faces = []

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("v "):
                parts = line.split()
                if len(parts) >= 4:
                    verts.append([float(parts[1]), float(parts[2]), float(parts[3])])
            elif line.startswith("f "):
                parts = line.split()[1:]
                idx = []
                for p in parts:
                    # supports formats like '1', '1/2', '1/2/3', '1//3'
                    v_str = p.split("/")[0]
                    if v_str:
                        idx.append(int(v_str) - 1)
                # triangulate fan if needed
                if len(idx) >= 3:
                    for k in range(1, len(idx) - 1):
                        faces.append([idx[0], idx[k], idx[k + 1]])

    verts = np.asarray(verts, dtype=np.float32)
    faces = np.asarray(faces, dtype=np.int32)
    return verts, faces

normal_verts, normal_faces = load_obj_vertices_faces(obj_path)
print("Normal mesh verts:", normal_verts.shape, "faces:", normal_faces.shape)
print("Normal verts sample:\n", normal_verts[:5])
print("Normal faces sample:\n", normal_faces[:5])

# Basic bounds sanity check
mins = normal_verts.min(axis=0)
maxs = normal_verts.max(axis=0)
print("Normal mesh bounds min:", mins, "max:", maxs, "extent:", (maxs - mins))


OBJ path: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/SubTool-0-7412864.OBJ
Exists: True
Normal mesh verts: (1501756, 3) faces: (3003486, 3)
Normal verts sample:
 [[ 0.27118     0.32905057 -0.04217894]
 [ 0.27177083  0.32905975 -0.042197  ]
 [ 0.26937002  0.32847005 -0.04219494]
 [ 0.26989704  0.32840753 -0.04210592]
 [ 0.27048913  0.32836676 -0.04204723]]
Normal faces sample:
 [[  0 162   4]
 [  0   4   5]
 [  1   0   5]
 [  1   5   6]
 [  7 163   1]]
Normal mesh bounds min: [-0.21345729 -0.49756598 -0.44434476] max: [0.5041339 0.6030878 0.7672508] extent: [0.71759117 1.1006538  1.2115955 ]


In [6]:
# Prep for ICP: choose a consistent anatomical label mesh per patient and downsample both patient + normal point clouds
# - The normal OBJ is extremely dense (~1.5M verts); we downsample it for fast ICP.
# - For patient segmentations, we pick a label (default: 3.0 if present else smallest non-zero label) and sample points.

from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(0)

# Parameters (tune as needed)
NORMAL_SAMPLE_N = 60000   # downsampled normal point cloud for ICP target
PATIENT_SAMPLE_N = 15000  # downsampled patient point cloud for ICP source
DEFAULT_LABEL = 3.0

# Build downsampled normal point cloud
normal_n = normal_verts.shape[0]
if normal_n <= NORMAL_SAMPLE_N:
    normal_pts = normal_verts.astype(np.float32)
else:
    idx = rng.choice(normal_n, size=NORMAL_SAMPLE_N, replace=False)
    normal_pts = normal_verts[idx].astype(np.float32)

print("Normal verts:", normal_verts.shape, "Downsampled pts:", normal_pts.shape)

# Helper to get a patient point cloud for ICP from stored meshes

def get_patient_pts_for_icp(pat: int, default_label: float = DEFAULT_LABEL, n_sample: int = PATIENT_SAMPLE_N):
    meshes = pat_meshes[pat]
    if default_label in meshes:
        v = meshes[default_label][0]
        used_label = default_label
    else:
        labs = sorted(meshes.keys())
        used_label = labs[0]
        v = meshes[used_label][0]

    if v.shape[0] <= n_sample:
        pts = v.astype(np.float32)
    else:
        idx = rng.choice(v.shape[0], size=n_sample, replace=False)
        pts = v[idx].astype(np.float32)
    return pts, used_label

# Quick sanity: show a few patients' selected labels and point counts
for pat in sorted(pat_meshes.keys())[:5]:
    pts, lab = get_patient_pts_for_icp(pat)
    print(f"Pat {pat}: using label {lab} | pts {pts.shape}")


Normal verts: (1501756, 3) Downsampled pts: (60000, 3)
Pat 0: using label 3.0 | pts (4626, 3)
Pat 1: using label 3.0 | pts (4418, 3)
Pat 2: using label 3.0 | pts (6422, 3)
Pat 3: using label 3.0 | pts (2341, 3)
Pat 4: using label 3.0 | pts (611, 3)


In [7]:
# Implement rigid ICP (rotation+translation) using point-to-point SVD (Kabsch)
import numpy as np
from sklearn.neighbors import NearestNeighbors

def rigid_transform_kabsch(A: np.ndarray, B: np.ndarray):
    """Find R,t s.t. A@R.T + t ~= B. A,B: (N,3) with correspondences."""
    assert A.shape == B.shape and A.shape[1] == 3
    centroid_A = A.mean(axis=0)
    centroid_B = B.mean(axis=0)
    AA = A - centroid_A
    BB = B - centroid_B
    H = AA.T @ BB
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    # Ensure right-handed rotation (det=+1)
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T
    t = centroid_B - centroid_A @ R.T
    return R.astype(np.float32), t.astype(np.float32)


def icp_rigid(source_pts: np.ndarray, target_pts: np.ndarray, max_iter: int = 50, tol: float = 1e-5):
    """Rigid ICP aligning source->target. Returns R,t and history."""
    src = source_pts.astype(np.float32).copy()
    tgt = target_pts.astype(np.float32)

    nn = NearestNeighbors(n_neighbors=1, algorithm="auto").fit(tgt)

    R_total = np.eye(3, dtype=np.float32)
    t_total = np.zeros(3, dtype=np.float32)

    prev_rmse = np.inf
    history = []

    for it in range(max_iter):
        dists, idx = nn.kneighbors(src, return_distance=True)
        matched = tgt[idx[:, 0]]
        rmse = float(np.sqrt((dists[:, 0] ** 2).mean()))
        history.append(rmse)

        R, t = rigid_transform_kabsch(src, matched)
        src = src @ R.T + t  # update source

        # Compose transforms: new_total(x) = (x@R_total.T + t_total)@R.T + t
        R_total = R @ R_total
        t_total = t + (t_total @ R.T)

        if abs(prev_rmse - rmse) < tol:
            break
        prev_rmse = rmse

    return R_total, t_total, history

# Quick QC: run ICP for a few patients
qc_pats = sorted(pat_meshes.keys())[:3]
for pat in qc_pats:
    src_pts, used_label = get_patient_pts_for_icp(pat)
    R, t, hist = icp_rigid(src_pts, normal_pts, max_iter=40, tol=1e-5)
    print(f"Pat {pat} | label {used_label} | iters {len(hist)} | rmse start {hist[0]:.3f} -> end {hist[-1]:.3f}")


Pat 0 | label 3.0 | iters 6 | rmse start 148.694 -> end 36.164
Pat 1 | label 3.0 | iters 8 | rmse start 123.460 -> end 30.642
Pat 2 | label 3.0 | iters 7 | rmse start 153.075 -> end 39.336


In [8]:
# Compute normal->aligned-patient nearest-neighbor distances for a small QC subset
from sklearn.neighbors import NearestNeighbors

QC_PATS = sorted(pat_meshes.keys())[:5]

qc_distance_maps = {}  # pat -> distances (len(normal_pts)) for quick sanity
qc_rmse = {}

# Fit NN on each aligned patient point cloud (source) and query from normal points
for pat in QC_PATS:
    src_pts, used_label = get_patient_pts_for_icp(pat)
    R, t, hist = icp_rigid(src_pts, normal_pts, max_iter=40, tol=1e-5)
    aligned_src = src_pts @ R.T + t

    nn = NearestNeighbors(n_neighbors=1, algorithm="auto").fit(aligned_src)
    dists, _ = nn.kneighbors(normal_pts, return_distance=True)
    d = dists[:, 0].astype(np.float32)

    qc_distance_maps[pat] = d
    qc_rmse[pat] = (used_label, hist[0], hist[-1], float(d.mean()), float(np.median(d)), float(d.max()))

qc_rmse_df = pd.DataFrame(
    [(p,)+qc_rmse[p] for p in QC_PATS],
    columns=["Pat","used_label","icp_rmse_start","icp_rmse_end","dist_mean","dist_median","dist_max"],
)

display(qc_rmse_df)
print("Normal query points:", normal_pts.shape[0])
print("Distance dtype/range example (first QC pat):", QC_PATS[0], qc_distance_maps[QC_PATS[0]].dtype, float(qc_distance_maps[QC_PATS[0]].min()), float(qc_distance_maps[QC_PATS[0]].max()))


,Pat,used_label,icp_rmse_start,icp_rmse_end,dist_mean,dist_median,dist_max
0,0,3.0,148.694366,36.164113,17.592106,17.553061,18.293739
1,1,3.0,123.459837,30.641891,10.794310,10.774761,11.416569
2,2,3.0,153.074855,39.336114,14.367657,14.348886,14.985520
3,3,3.0,183.704329,12.365194,5.249067,5.249059,5.603137
4,4,3.0,83.087697,7.186923,2.552636,2.573610,3.239522


Normal query points: 60000
Distance dtype/range example (first QC pat): 0 float32 16.924365997314453 18.293739318847656


In [9]:
# Compute and save per-patient distance maps: normal_pts -> aligned patient (nearest-neighbor)
# Outputs:
# - distance_maps/{pat:03d}_normal_to_patient_dist.npy (len=NORMAL_SAMPLE_N)
# - per_patient_distance_stats.csv

from pathlib import Path
from sklearn.neighbors import NearestNeighbors

OUT_DIR = PROJECT_ROOT / "outputs"
DIST_DIR = OUT_DIR / "distance_maps"
DIST_DIR.mkdir(parents=True, exist_ok=True)

# We'll use the same normal_pts created earlier (downsampled normal vertices)
assert isinstance(normal_pts, np.ndarray) and normal_pts.ndim == 2 and normal_pts.shape[1] == 3

per_pat_rows = []

# Note: this is O(#patients * (ICP + NN queries)). Should be manageable at this sampling level.
for pat in sorted(pat_meshes.keys()):
    src_pts, used_label = get_patient_pts_for_icp(pat)
    R, t, hist = icp_rigid(src_pts, normal_pts, max_iter=40, tol=1e-5)
    aligned_src = src_pts @ R.T + t

    nn_pat = NearestNeighbors(n_neighbors=1, algorithm="auto").fit(aligned_src)
    dists, _ = nn_pat.kneighbors(normal_pts, return_distance=True)
    d = dists[:, 0].astype(np.float32)

    # Save distance map for this patient
    out_path = DIST_DIR / f"pat{pat:03d}_normal_to_patient_dist.npy"
    np.save(out_path, d)

    per_pat_rows.append(
        {
            "Pat": int(pat),
            "used_label": float(used_label),
            "icp_iters": int(len(hist)),
            "icp_rmse_start": float(hist[0]),
            "icp_rmse_end": float(hist[-1]),
            "dist_mean": float(d.mean()),
            "dist_median": float(np.median(d)),
            "dist_max": float(d.max()),
            "dist_p95": float(np.quantile(d, 0.95)),
            "n_source_pts": int(src_pts.shape[0]),
            "n_normal_pts": int(normal_pts.shape[0]),
            "dist_npy": str(out_path.relative_to(PROJECT_ROOT)),
        }
    )

per_patient_stats = pd.DataFrame(per_pat_rows).sort_values("Pat").reset_index(drop=True)
per_patient_stats_path = OUT_DIR / "per_patient_distance_stats.csv"
per_patient_stats.to_csv(per_patient_stats_path, index=False)

display(per_patient_stats.head(10))
print("Saved per-patient distance maps to:", DIST_DIR)
print("Saved per-patient stats to:", per_patient_stats_path)
print("Patients:", per_patient_stats.shape[0], "| normal points per patient:", int(normal_pts.shape[0]))
print("dist_mean summary:\n", per_patient_stats["dist_mean"].describe())


,Pat,used_label,icp_iters,icp_rmse_start,icp_rmse_end,dist_mean,dist_median,dist_max,dist_p95,n_source_pts,n_normal_pts,dist_npy
0,0,3.0,6,148.694366,36.164113,17.592106,17.553061,18.293739,18.103247,4626,60000,outputs/distance_maps/pat000_normal_to_patient...
1,1,3.0,8,123.459837,30.641891,10.794310,10.774761,11.416569,11.270830,4418,60000,outputs/distance_maps/pat001_normal_to_patient...
2,2,3.0,7,153.074855,39.336114,14.367657,14.348886,14.985520,14.862238,6422,60000,outputs/distance_maps/pat002_normal_to_patient...
3,3,3.0,40,183.704329,12.365194,5.249067,5.249059,5.603137,5.521433,2341,60000,outputs/distance_maps/pat003_normal_to_patient...
4,4,3.0,11,83.087697,7.186923,2.552636,2.573610,3.239522,3.044530,611,60000,outputs/distance_maps/pat004_normal_to_patient...
5,5,3.0,18,105.277008,13.108470,1.437639,1.396152,2.132100,1.960051,1884,60000,outputs/distance_maps/pat005_normal_to_patient...
6,6,3.0,11,111.367825,28.633376,13.756603,13.726612,14.416109,14.244706,5892,60000,outputs/distance_maps/pat006_normal_to_patient...
7,7,3.0,12,146.998503,32.863721,0.421752,0.400329,0.967100,0.750131,8829,60000,outputs/distance_maps/pat007_normal_to_patient...
8,8,3.0,8,128.214860,33.063233,13.745860,13.710653,14.470472,14.287150,4483,60000,outputs/distance_maps/pat008_normal_to_patient...
9,9,3.0,7,146.307136,39.326693,25.411062,25.452976,25.774927,25.676882,7142,60000,outputs/distance_maps/pat009_normal_to_patient...


Saved per-patient distance maps to: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/outputs/distance_maps
Saved per-patient stats to: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/outputs/per_patient_distance_stats.csv
Patients: 60 | normal points per patient: 60000
dist_mean summary:
 count    60.000000
mean      9.600400
std       7.637060
min       0.421752
25%       2.445355
50%       8.929782
75%      15.376094
max      28.768272
Name: dist_mean, dtype: float64


In [10]:
# Aggregate distance statistics per clinical disease label (columns with 'X')
from pathlib import Path

# Identify binary disease/condition columns in hvsmr_clinical (everything except these)
ID_COLS = {"Pat", "Age", "Category"}
disease_cols = [c for c in hvsmr_clinical.columns if c not in ID_COLS]

# Normalize Pat to int for join
clinical = hvsmr_clinical.copy()
clinical["Pat"] = pd.to_numeric(clinical["Pat"], errors="coerce").astype("Int64")

# Some Pats appear in distance maps but not clinical (e.g., 59). We'll restrict to Pats present in clinical.
valid_pats = set(clinical["Pat"].dropna().astype(int).tolist())

# Map Pat -> dist file path using per_patient_stats (already computed)
pps = per_patient_stats.copy()
pps["Pat"] = pps["Pat"].astype(int)
pps = pps[pps["Pat"].isin(valid_pats)].reset_index(drop=True)

pat_to_npy = {int(r.Pat): (PROJECT_ROOT / r.dist_npy) for r in pps.itertuples(index=False)}

# Helper: aggregate over patients with clinical[col] == 'X'
rows = []
for col in disease_cols:
    pats_with_label = clinical.loc[clinical[col].astype(str).str.upper().eq("X"), "Pat"].dropna().astype(int).tolist()
    pats_with_label = [p for p in pats_with_label if p in pat_to_npy]

    if len(pats_with_label) == 0:
        rows.append({"label": col, "n_patients": 0, "mean_of_means": np.nan, "median_of_medians": np.nan, "max_of_max": np.nan})
        continue

    # Patient-level summaries
    means, medians, maxs = [], [], []
    for p in pats_with_label:
        d = np.load(pat_to_npy[p])
        means.append(float(d.mean()))
        medians.append(float(np.median(d)))
        maxs.append(float(d.max()))

    rows.append(
        {
            "label": col,
            "n_patients": int(len(pats_with_label)),
            "mean_of_means": float(np.mean(means)),
            "median_of_medians": float(np.median(medians)),
            "max_of_max": float(np.max(maxs)),
        }
    )

per_disease_summary = pd.DataFrame(rows).sort_values(["n_patients", "mean_of_means"], ascending=[False, False]).reset_index(drop=True)

# Save
per_disease_path = OUT_DIR / "per_disease_distance_summary.csv"
per_disease_summary.to_csv(per_disease_path, index=False)

display(per_disease_summary.head(20))
print("Saved per-disease summary to:", per_disease_path)
print("Non-empty labels:", int((per_disease_summary["n_patients"] > 0).sum()), "/", per_disease_summary.shape[0])


,label,n_patients,mean_of_means,median_of_medians,max_of_max
0,VSD,29,8.941929,8.402134,25.774927
1,Glenn,24,8.806135,8.476928,19.929501
2,ASD,22,7.437846,5.706434,19.929501
3,DORV,22,7.420508,5.687853,18.980591
4,LeftCentralIVC,16,7.880395,6.021553,19.929501
5,InvertedVentricles,15,8.406043,8.551722,25.774927
6,Heterotaxy,13,6.192006,2.583418,19.310246
7,CMRArtifactPA,13,5.623976,3.640972,18.980591
8,Dextrocardia,10,12.401783,13.813716,25.774927
9,CommonAtrium,10,6.167177,3.112195,19.310246


Saved per-disease summary to: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/outputs/per_disease_distance_summary.csv
Non-empty labels: 33 / 33


In [11]:
# Export a few representative patients as colorized point-cloud meshes (PLY)
# - Uses normal_pts (downsampled normal reference vertices)
# - Colors each normal point by its distance to the aligned patient (from saved .npy)
# - Writes ASCII PLY with per-vertex RGB

from pathlib import Path

COLOR_DIR = OUT_DIR / "colorized_meshes"
COLOR_DIR.mkdir(parents=True, exist_ok=True)

# Pick representative patients: low/median/high dist_mean among those with clinical entries
pps_valid = per_patient_stats[per_patient_stats["Pat"].isin(valid_pats)].copy()
pps_valid = pps_valid.sort_values("dist_mean").reset_index(drop=True)

rep_pats = {
    "low": int(pps_valid.iloc[0].Pat),
    "median": int(pps_valid.iloc[len(pps_valid)//2].Pat),
    "high": int(pps_valid.iloc[-1].Pat),
}
print("Representative Pats:", rep_pats)

def _dist_to_rgb(dist: np.ndarray, vmin: float, vmax: float):
    """Map distances to RGB using a simple blue->red gradient."""
    x = (dist - vmin) / (vmax - vmin + 1e-12)
    x = np.clip(x, 0.0, 1.0)
    # blue (low) -> red (high)
    r = (255.0 * x).astype(np.uint8)
    g = (255.0 * (1.0 - np.abs(x - 0.5) * 2.0)).astype(np.uint8)  # green in middle
    b = (255.0 * (1.0 - x)).astype(np.uint8)
    return r, g, b

def write_ply_pointcloud(path: Path, xyz: np.ndarray, rgb: np.ndarray):
    assert xyz.shape[0] == rgb.shape[0] and xyz.shape[1] == 3 and rgb.shape[1] == 3
    n = xyz.shape[0]
    with open(path, "w", encoding="utf-8") as f:
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"element vertex {n}\n")
        f.write("property float x\nproperty float y\nproperty float z\n")
        f.write("property uchar red\nproperty uchar green\nproperty uchar blue\n")
        f.write("end_header\n")
        for i in range(n):
            x, y, z = xyz[i]
            r, g, b = rgb[i]
            f.write(f"{x:.6f} {y:.6f} {z:.6f} {int(r)} {int(g)} {int(b)}\n")

# Use a global color scale based on per-patient stats (p95 as vmax to reduce outlier influence)
vmin = 0.0
vmax = float(np.quantile(pps_valid["dist_p95"].values, 0.9))
print("Color scale vmin/vmax:", vmin, vmax)

exports = []
for tag, pat in rep_pats.items():
    # Load saved per-vertex distances (for normal_pts)
    dist_path = PROJECT_ROOT / pps_valid.loc[pps_valid["Pat"].eq(pat), "dist_npy"].iloc[0]
    d = np.load(dist_path).astype(np.float32)
    assert d.shape[0] == normal_pts.shape[0]

    r, g, b = _dist_to_rgb(d, vmin=vmin, vmax=vmax)
    rgb = np.stack([r, g, b], axis=1)

    out_ply = COLOR_DIR / f"pat{pat:03d}_{tag}_normalPts_coloredByDist.ply"
    write_ply_pointcloud(out_ply, normal_pts.astype(np.float32), rgb)
    exports.append((tag, pat, out_ply))

print("Wrote colorized PLY point-clouds:")
for tag, pat, pth in exports:
    print(f" - {tag:6s} pat{pat:03d}: {pth}")


Representative Pats: {'low': 7, 'median': 42, 'high': 27}
Color scale vmin/vmax: 0.0 18.862404117584227
Wrote colorized PLY point-clouds:
 - low    pat007: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/outputs/colorized_meshes/pat007_low_normalPts_coloredByDist.ply
 - median pat042: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/outputs/colorized_meshes/pat042_median_normalPts_coloredByDist.ply
 - high   pat027: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/outputs/colorized_meshes/pat027_high_normalPts_coloredByDist.ply


In [12]:
# Overlay visualizer: normal (white) + aligned patient (colored by distance)
# - Uses the SAME point clouds for ICP, distance, and display
# - Rigid-only: rotation + translation (no scaling)
# - Colors patient points by patient->normal distance (blue=close, red=far)

from pathlib import Path
import numpy as np
from sklearn.neighbors import NearestNeighbors

# Plotly
try:
    import plotly.graph_objects as go
except ModuleNotFoundError:
    !pip -q install plotly
    import plotly.graph_objects as go


def _center(xyz: np.ndarray):
    xyz = xyz.astype(np.float32)
    c = xyz.mean(axis=0)
    return xyz - c, c


def _dist_to_rgb(dist: np.ndarray, vmin: float = 0.0, vmax: float = None):
    if vmax is None:
        vmax = float(np.quantile(dist, 0.95))
    x = np.clip((dist - vmin) / (vmax - vmin + 1e-12), 0.0, 1.0)
    r = (255.0 * x).astype(np.uint8)
    g = (255.0 * (1.0 - np.abs(x - 0.5) * 2.0)).astype(np.uint8)
    b = (255.0 * (1.0 - x)).astype(np.uint8)
    colors = [f"rgb({ri},{gi},{bi})" for ri, gi, bi in zip(r, g, b)]
    return colors, vmax


def make_overlay_html(
    pat: int,
    out_html: Path,
    max_patient_points: int = 25000,
    seed: int = 0,
    normal_point_size: float = 2.0,
    patient_point_size: float = 2.0,
    patient_opacity: float = 0.95,
    vmax_quantile: float = 0.95,
    center_only_for_view: bool = True,
):
    """Write an overlay HTML comparing normal vs aligned patient.

    - Normal points: white
    - Patient points: colored by patient->normal distance (blue=close, red=far)
    - Rigid ICP only; no scaling
    """
    rng_local = np.random.default_rng(seed)

    # --- 1) Patient points (same points for ICP + display) ---
    src_pts_full, used_label = get_patient_pts_for_icp(pat)
    if src_pts_full.shape[0] > max_patient_points:
        idx = rng_local.choice(src_pts_full.shape[0], size=max_patient_points, replace=False)
        src_pts = src_pts_full[idx]
    else:
        src_pts = src_pts_full

    # --- 2) Use the SAME normal points already used for distance maps ---
    normal_for_icp = normal_pts.astype(np.float32)
    src_for_icp = src_pts.astype(np.float32)

    # --- 3) Rigid ICP align patient->normal ---
    R, t, hist = icp_rigid(src_for_icp, normal_for_icp, max_iter=60, tol=1e-6)
    aligned_src = src_for_icp @ R.T + t

    # --- 4) Distances for coloring (patient->normal) ---
    nn_norm = NearestNeighbors(n_neighbors=1).fit(normal_for_icp)
    d_p2n, _ = nn_norm.kneighbors(aligned_src, return_distance=True)
    d_p2n = d_p2n[:, 0].astype(np.float32)
    colors_p2n, vmax_used = _dist_to_rgb(d_p2n, vmin=0.0, vmax=float(np.quantile(d_p2n, vmax_quantile)))

    # --- 5) Optional centering for prettier overlay (no scaling) ---
    if center_only_for_view:
        normal_plot, _ = _center(normal_for_icp)
        patient_plot, _ = _center(aligned_src)
    else:
        normal_plot = normal_for_icp
        patient_plot = aligned_src

    # --- 6) Build figure ---
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=normal_plot[:, 0],
                y=normal_plot[:, 1],
                z=normal_plot[:, 2],
                mode="markers",
                name="Normal (white)",
                marker=dict(size=normal_point_size, color="rgb(255,255,255)", opacity=0.90),
            ),
            go.Scatter3d(
                x=patient_plot[:, 0],
                y=patient_plot[:, 1],
                z=patient_plot[:, 2],
                mode="markers",
                name=f"Patient {pat} (colored by distance)",
                marker=dict(size=patient_point_size, color=colors_p2n, opacity=patient_opacity),
            ),
        ]
    )

    fig.update_layout(
        title=(
            f"Overlay: Pat {pat} vs Normal | label={used_label} | ICP iters={len(hist)} rmse_end={hist[-1]:.3f} | "
            f"color scale: blue→red (vmax q={vmax_quantile}, vmax={vmax_used:.3f})"
        ),
        scene=dict(aspectmode="data"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(l=0, r=0, b=0, t=70),
        height=760,
        paper_bgcolor="rgb(0,0,0)",
    )

    out_html.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(str(out_html), include_plotlyjs="cdn")

    return {
        "pat": pat,
        "used_label": used_label,
        "icp_iters": len(hist),
        "icp_rmse_end": float(hist[-1]),
        "out_html": str(out_html),
        "vmax_used": float(vmax_used),
        "n_normal_pts": int(normal_for_icp.shape[0]),
        "n_patient_pts_rendered": int(patient_plot.shape[0]),
        "center_only_for_view": bool(center_only_for_view),
    }


# --- Create an overlay for a chosen patient ---
PAT_TO_VIEW = int(PAT_TO_VIEW) if "PAT_TO_VIEW" in globals() else 7
html_path = (OUT_DIR / "figures" / f"pat{PAT_TO_VIEW:03d}_overlay_white_normal_colored_patient.html")
info = make_overlay_html(PAT_TO_VIEW, html_path, center_only_for_view=True)

print("Wrote overlay viewer:")
print(" -", info["out_html"])
print("Details:", {k: info[k] for k in ["used_label", "icp_iters", "icp_rmse_end", "n_patient_pts_rendered"]})
print("\nOpen the HTML file in your browser to view the overlay.")


Wrote overlay viewer:
 - /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/outputs/figures/pat007_overlay_white_normal_colored_patient.html
Details: {'used_label': 3.0, 'icp_iters': 14, 'icp_rmse_end': 32.86371973469591, 'n_patient_pts_rendered': 8829}

Open the HTML file in your browser to view the overlay.


In [13]:
# Diagnose scale mismatch between normal reference (OBJ) and patient surfaces (segmentation-derived)
# Computes characteristic lengths (bounding box diagonal, RMS radius) for normal and a few patients.

import numpy as np


def cloud_stats(xyz: np.ndarray):
    xyz = np.asarray(xyz, dtype=np.float64)
    c = xyz.mean(axis=0)
    centered = xyz - c
    mins = xyz.min(axis=0)
    maxs = xyz.max(axis=0)
    extent = maxs - mins
    bbox_diag = float(np.linalg.norm(extent))
    rms_radius = float(np.sqrt((centered**2).sum(axis=1).mean()))
    p95_radius = float(np.quantile(np.sqrt((centered**2).sum(axis=1)), 0.95))
    return {
        "n": int(xyz.shape[0]),
        "mins": mins,
        "maxs": maxs,
        "extent": extent,
        "bbox_diag": bbox_diag,
        "rms_radius": rms_radius,
        "p95_radius": p95_radius,
        "centroid": c,
    }

# Normal cloud: use downsampled normal_pts (already in memory)
normal_s = cloud_stats(normal_pts)
print("NORMAL (downsampled) stats:")
print({k: normal_s[k] for k in ["n","bbox_diag","rms_radius","p95_radius","extent"]})

# Patient clouds: inspect a few Pats across dist_mean spectrum
# Prefer using the same point selection as ICP (label 3.0, sampled)
probe_pats = [int(pps_valid.iloc[0].Pat), int(pps_valid.iloc[len(pps_valid)//2].Pat), int(pps_valid.iloc[-1].Pat)]
print("\nProbe pats (low/median/high dist_mean among clinical pats):", probe_pats)

rows = []
for p in probe_pats:
    pts, lab = get_patient_pts_for_icp(p)
    s = cloud_stats(pts)
    rows.append((p, float(lab), s["bbox_diag"], s["rms_radius"], s["p95_radius"], s["extent"]))
    print(f"Pat {p:03d} (label {lab}) | bbox_diag={s['bbox_diag']:.3f} | rms_radius={s['rms_radius']:.3f} | p95_radius={s['p95_radius']:.3f} | extent={s['extent']}")

# Ratios (patient vs normal)
print("\nScale ratios (patient / normal):")
for (p, lab, bbox_diag, rms_radius, p95_radius, extent) in rows:
    print(
        f"Pat {p:03d}: bbox_diag ratio={bbox_diag/normal_s['bbox_diag']:.3f}, "
        f"rms_radius ratio={rms_radius/normal_s['rms_radius']:.3f}"
    )


NORMAL (downsampled) stats:
{'n': 60000, 'bbox_diag': 1.7850905885418193, 'rms_radius': 0.40389799712280516, 'p95_radius': 0.6384164717172075, 'extent': array([0.71731712, 1.09861496, 1.21039239])}

Probe pats (low/median/high dist_mean among clinical pats): [7, 42, 27]
Pat 007 (label 3.0) | bbox_diag=118.212 | rms_radius=33.416 | p95_radius=61.755 | extent=[44.59471893 62.87124252 89.625     ]
Pat 042 (label 3.0) | bbox_diag=84.823 | rms_radius=25.549 | p95_radius=35.892 | extent=[33.046875   27.421875   73.14968109]
Pat 027 (label 3.0) | bbox_diag=118.933 | rms_radius=41.597 | p95_radius=53.920 | extent=[42.1875     52.734375   97.89919281]

Scale ratios (patient / normal):
Pat 007: bbox_diag ratio=66.222, rms_radius ratio=82.733
Pat 042: bbox_diag ratio=47.517, rms_radius ratio=63.255
Pat 027: bbox_diag ratio=66.625, rms_radius ratio=102.988


## Notebook cleanup + what “normalization” means

### What was removed
Earlier we had several exploratory / debugging visualization cells (extra static plots, multiple PLY re-exports with different color scales, and ad‑hoc diagnostics). Those have been removed to keep this notebook focused on the core pipeline + one reusable viewer.

### What remains (core deliverables)
1. Load `hvsmr_clinical.csv` + `hvsmr_technical.csv`
2. Map `Pat → *_cropped_seg_endpoints.nii.gz`
3. Marching-cubes surface extraction per patient
4. Load the normal reference OBJ mesh
5. Rigid ICP (rotation + translation)
6. Compute and save distance maps (normal → patient NN distances)
7. Aggregate per-disease summaries
8. Export representative colorized point-cloud PLYs
9. **Interactive overlay viewer** (`make_overlay_html`) to view *both* shapes at once

### Important: why the overlay looked “way off” in size
- The **patient** surfaces are derived from NIfTI segmentations and use voxel spacing; they are effectively in **millimeters** (or mm-like units).
- The **normal reference OBJ** appears to be in a **different / arbitrary unit scale** (extent ~1–2 units).

Because ICP in this notebook is **rigid only** (no scaling), it can align orientation/position but it **cannot reconcile unit mismatches**. That is why the overlay can look badly scaled.

### What we did for visualization
The overlay viewer cell now supports an optional **centering + isotropic scaling normalization** (controlled by `normalize_scale=True` and `scale_method`).

- If `normalize_scale=False`: distances are in the patient’s native units, but the overlay may look mismatched in size.
- If `normalize_scale=True`: both clouds are normalized to comparable “unit size” so the overlay is visually meaningful, **but the displayed distances are in normalized units (not mm)**.

If you want *physically meaningful* mm distances **and** a correct overlay, we need the true scale relationship between the OBJ and the segmentations (e.g., the OBJ’s unit→mm conversion).

In [14]:
# Enable inline Plotly rendering in this notebook (installs nbformat if missing)
import importlib
import sys

try:
    import nbformat  # noqa: F401
except ModuleNotFoundError:
    !pip -q install nbformat
    import nbformat  # noqa: F401

# If you're in JupyterLab/VSCode, this typically enables inline figures
import plotly.io as pio
pio.renderers.default = "notebook_connected"

print("Plotly renderer set to:", pio.renderers.default)

Plotly renderer set to: notebook_connected


In [15]:
# Render the NORMAL reference point cloud by itself (inline Plotly)
import numpy as np
import plotly.graph_objects as go

NORMAL_POINT_SIZE = 2.0
CENTER_ONLY_FOR_VIEW = True


def _center(xyz: np.ndarray):
    xyz = xyz.astype(np.float32)
    c = xyz.mean(axis=0)
    return xyz - c, c


normal_view = normal_pts.astype(np.float32)
if CENTER_ONLY_FOR_VIEW:
    normal_view, _ = _center(normal_view)

fig_normal = go.Figure(
    data=[
        go.Scatter3d(
            x=normal_view[:, 0],
            y=normal_view[:, 1],
            z=normal_view[:, 2],
            mode="markers",
            name="Normal reference",
            marker=dict(size=NORMAL_POINT_SIZE, color="rgb(255,255,255)", opacity=0.90),
        )
    ]
)
fig_normal.update_layout(
    title=f"Normal reference point cloud (n={normal_view.shape[0]}) | centered={CENTER_ONLY_FOR_VIEW}",
    scene=dict(aspectmode="data"),
    margin=dict(l=0, r=0, b=0, t=60),
    height=720,
    paper_bgcolor="rgb(0,0,0)",
)

fig_normal.show()


In [16]:
# Render a PATIENT point cloud by itself (aligned to the normal reference; inline Plotly)
import numpy as np
import plotly.graph_objects as go
from sklearn.neighbors import NearestNeighbors

PAT_TO_RENDER = 7  # change this
CENTER_ONLY_FOR_VIEW = True
PATIENT_POINT_SIZE = 2.0


def _center(xyz: np.ndarray):
    xyz = xyz.astype(np.float32)
    c = xyz.mean(axis=0)
    return xyz - c, c


def _dist_to_rgb(dist: np.ndarray, vmin: float = 0.0, vmax: float = None):
    if vmax is None:
        vmax = float(np.quantile(dist, 0.95))
    x = np.clip((dist - vmin) / (vmax - vmin + 1e-12), 0.0, 1.0)
    r = (255.0 * x).astype(np.uint8)
    g = (255.0 * (1.0 - np.abs(x - 0.5) * 2.0)).astype(np.uint8)
    b = (255.0 * (1.0 - x)).astype(np.uint8)
    colors = [f"rgb({ri},{gi},{bi})" for ri, gi, bi in zip(r, g, b)]
    return colors, vmax


# Prepare clouds in the same space used for ICP
src_pts_full, used_label = get_patient_pts_for_icp(int(PAT_TO_RENDER))
normal_for_icp = normal_pts.astype(np.float32)

# Align patient -> normal
R, t, hist = icp_rigid(src_pts_full.astype(np.float32), normal_for_icp, max_iter=60, tol=1e-6)
patient_aligned = src_pts_full.astype(np.float32) @ R.T + t

# Distances patient->normal for coloring
nn_norm = NearestNeighbors(n_neighbors=1).fit(normal_for_icp)
d_p2n, _ = nn_norm.kneighbors(patient_aligned, return_distance=True)
d_p2n = d_p2n[:, 0].astype(np.float32)
colors, vmax_used = _dist_to_rgb(d_p2n, vmin=0.0, vmax=None)

if CENTER_ONLY_FOR_VIEW:
    patient_aligned, _ = _center(patient_aligned)

fig_pat = go.Figure(
    data=[
        go.Scatter3d(
            x=patient_aligned[:, 0],
            y=patient_aligned[:, 1],
            z=patient_aligned[:, 2],
            mode="markers",
            name=f"Patient {PAT_TO_RENDER} (aligned)",
            marker=dict(size=PATIENT_POINT_SIZE, color=colors, opacity=0.95),
        )
    ]
)
fig_pat.update_layout(
    title=(
        f"Patient {PAT_TO_RENDER} point cloud (aligned) | label={used_label} | "
        f"ICP iters={len(hist)} rmse_end={hist[-1]:.4f} | colored by distance (vmax={vmax_used:.3f})"
    ),
    scene=dict(aspectmode="data"),
    margin=dict(l=0, r=0, b=0, t=70),
    height=720,
    paper_bgcolor="rgb(0,0,0)",
)

fig_pat.show()
